## LOG AND REGISTER WITH ML-FLOW

In [0]:
%run "/Workspace/HRPolicyAgent/Build_HR_PolicyAgent"

#### LOG AGENT AS ML FLOW AGENT

In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.models.resources import (
    DatabricksVectorSearchIndex,
    DatabricksServingEndpoint
)
import langchain_core

# ── CRITICAL: Tell MLflow what to log ─────────────────────────────────────────
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/hr-policy-agent")

# ── Enable autologging for full tracing of every invocation ───────────────────
mlflow.langchain.autolog()

# ── Input example so MLflow can infer the model signature ─────────────────────
input_example = {"input": "How many annual leave days am I entitled to?", "current_user": "saikat.sengupta@mssquare.com"}

response = agent_executor.invoke(input_example)
signature = infer_signature(input_example, response)

resources = [
    DatabricksVectorSearchIndex(
        index_name="workspace.hr.policy_chunks_index"
    ),
    DatabricksServingEndpoint(
        endpoint_name="databricks-meta-llama-3-3-70b-instruct"
    ),
    DatabricksServingEndpoint(
        endpoint_name="databricks-bge-large-en"           # embedding model
    )
]

# ── Log using Models from Code — point to agent.py, not an object ─────────────
with mlflow.start_run(run_name="hr_policy_agent_v8"):
    logged_agent_info = mlflow.langchain.log_model(
        lc_model="/Workspace/HRPolicyAgent/Build_HR_PolicyAgent",  # path to Notebook 1
        artifact_path="agent",
        input_example=input_example,
        signature= signature,
        registered_model_name="hr.hr_policy_agent",  # UC registration
        resources = resources,
        pip_requirements=[
            "langchain>=0.3.0,<1.0.0",
            "langchain-core>=0.3.0,<1.0.0",
            "langchain-community>=0.3.0,<1.0.0",
            "databricks-langchain>=0.3.0,<1.0.0",
            "mlflow>=2.12.0"
        ]
    )

print(f"Run ID  : {logged_agent_info.run_id}")
print(f"Model URI: {logged_agent_info.model_uri}")


# # ── Verify it works before deploying ──────────────────────────────────────────
# loaded_agent = mlflow.langchain.load_model(logged_agent_info.model_uri)

# response = loaded_agent.invoke(
#     {"current_user":"saikat.sengupta@mssquare.com", "input": "What is the leave policy?"}
# )